In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set visualization style for all plots
sns.set_theme(style="whitegrid", palette="muted")

# Define the directory where your 17 CSV files are stored
DATA_DIR = '../data/PRO-ACT_Data/2026_02_27_PROACT_ALL_FORMS/' 

# ---------------------------------------------------------
# 1. DATA LOADING
# ---------------------------------------------------------
def load_data(filename):
    """Safely load a CSV file into a pandas DataFrame."""
    file_path = os.path.join(DATA_DIR, filename)
    try:
        print(f"Loading {filename}...")
        return pd.read_csv(file_path, low_memory=False)
    except FileNotFoundError:
        print(f"  -> Warning: {filename} not found. Skipping.")
        return None


In [2]:
print("=== PART 1: LOADING DATASETS ===")
datasets = {
    'demographics': load_data('F_PROACT_DEMOGRAPHICS.csv'),
    'alsfrs': load_data('F_PROACT_ALSFRS.csv'),
    'death': load_data('F_PROACT_DEATHDATA.csv'),
    'treatment': load_data('F_PROACT_TREATMENT.csv'),
    'riluzole': load_data('F_PROACT_RILUZOLE.csv'),
    'fvc': load_data('F_PROACT_FVC.csv'),
    'svc': load_data('F_PROACT_SVC.csv'),
    'vitals': load_data('F_PROACT_VITALSIGNS.csv'),
    'handgrip': load_data('F_PROACT_HANDGRIPSTRENGTH.csv'),
    'muscle': load_data('F_PROACT_MUSCLESTRENGTH.csv'),
    'neurofilament': load_data('F_PROACT_Neurofilament.csv'),
    'adverse_events': load_data('F_PROACT_ADVERSEEVENTS.csv'),
    'history': load_data('F_PROACT_ALSHISTORY.csv'),
    'conmeds': load_data('F_PROACT_CONMEDS.csv'),
    'elescorial': load_data('F_PROACT_ELESCORIAL.csv'),
    'family': load_data('F_PROACT_FAMILYHISTORY.csv'),
    'labs': load_data('F_PROACT_LABS.csv')
}

=== PART 1: LOADING DATASETS ===
Loading F_PROACT_DEMOGRAPHICS.csv...
Loading F_PROACT_ALSFRS.csv...
Loading F_PROACT_DEATHDATA.csv...
Loading F_PROACT_TREATMENT.csv...
Loading F_PROACT_RILUZOLE.csv...
Loading F_PROACT_FVC.csv...
Loading F_PROACT_SVC.csv...
Loading F_PROACT_VITALSIGNS.csv...
Loading F_PROACT_HANDGRIPSTRENGTH.csv...
Loading F_PROACT_MUSCLESTRENGTH.csv...
Loading F_PROACT_Neurofilament.csv...
Loading F_PROACT_ADVERSEEVENTS.csv...
Loading F_PROACT_ALSHISTORY.csv...
Loading F_PROACT_CONMEDS.csv...
Loading F_PROACT_ELESCORIAL.csv...
Loading F_PROACT_FAMILYHISTORY.csv...
Loading F_PROACT_LABS.csv...


In [ ]:
import pandas as pd
import numpy as np

def preprocess_proact_for_irt(dfs):
    """
    Preprocesses the PRO-ACT dataset dictionary for a Continuous-Time IRT/DFA-OU model.
    """
    
    # -------------------------------------------------------------------------
    # 1. PROCESS LONGITUDINAL MEASUREMENTS (Y) & DYNAMIC COVARIATES (x^y)
    # -------------------------------------------------------------------------
    df_alsfrs = dfs['alsfrs'].copy()
    
    # Derive time-varying Gastrostomy Status (x^y for Item 5 DIF)
    # If Q5b is not null, they have a gastrostomy.
    df_alsfrs['Gastrostomy_Status'] = df_alsfrs['Q5b_Cutting_with_Gastrostomy'].notna().astype(int)
    
    # Combine Q5a and Q5b into a single unified Q5 column
    df_alsfrs['Q5_Cutting'] = df_alsfrs['Q5b_Cutting_with_Gastrostomy'].fillna(df_alsfrs['Q5a_Cutting_without_Gastrostomy'])
    
    # Define the 12 ALSFRS-R items
    item_cols = [
        'Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing', 'Q4_Handwriting',
        'Q5_Cutting', 'Q6_Dressing_and_Hygiene', 'Q7_Turning_in_Bed',
        'Q8_Walking', 'Q9_Climbing_Stairs', 'R_1_Dyspnea',
        'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency'
    ]
    
    # Filter to essential columns and drop rows with missing time (Delta)
    df_model = df_alsfrs[['subject_id', 'ALSFRS_Delta', 'Gastrostomy_Status'] + item_cols].copy()
    df_model = df_model.dropna(subset=['ALSFRS_Delta'])
    
    # -------------------------------------------------------------------------
    # 2. PROCESS STATIC MEASUREMENT COVARIATES (x^y)
    # -------------------------------------------------------------------------
    df_demo = dfs['demographics'].copy()
    
    # Map Sex to binary (Female = 1, Male = 0)
    df_demo['Sex_Female'] = (df_demo['Sex'] == 'Female').astype(float)
    
    # Aggregate baseline demographics per subject
    df_demo_clean = df_demo.groupby('subject_id')[['Age', 'Sex_Female']].first().reset_index()
    
    # -------------------------------------------------------------------------
    # 3. PROCESS STRUCTURAL COVARIATES (x^xi)
    # -------------------------------------------------------------------------
    df_hist = dfs['history'].copy()
    
    # [Previous code...] Site of Onset and Diagnostic Delay
    if 'Site_of_Onset___Bulbar' in df_hist.columns:
        df_hist['Bulbar_Onset'] = df_hist['Site_of_Onset___Bulbar'].fillna(0)
    else:
        df_hist['Bulbar_Onset'] = (df_hist['Site_of_Onset'] == 'Bulbar').astype(float)
        
    df_hist['Diagnostic_Delay'] = df_hist['Diagnosis_Delta'] - df_hist['Onset_Delta']
    
    # Get the earliest Onset_Delta per patient
    df_hist_clean = df_hist.groupby('subject_id')[['Bulbar_Onset', 'Diagnostic_Delay', 'Onset_Delta']].first().reset_index()
    
    # Merge with Demographics to calculate Age at Onset
    # Age is at Time 0. Onset_Delta is negative days. 
    # Example: Age 60, Onset_Delta -365 days -> Age at Onset = 59
    df_hist_clean = df_hist_clean.merge(df_demo_clean[['subject_id', 'Age']], on='subject_id', how='left')
    df_hist_clean['Age_at_Onset'] = df_hist_clean['Age'] + (df_hist_clean['Onset_Delta'] / 365.25)
    
    # Drop the intermediate columns if desired
    df_hist_clean = df_hist_clean.drop(columns=['Onset_Delta', 'Age'])
    
    # -------------------------------------------------------------------------
    # 4. PROCESS BASELINE BIOMARKERS (x^xi)
    # -------------------------------------------------------------------------
    
    # Baseline Uric Acid
    df_labs = dfs['labs'].copy()
    uric = df_labs[df_labs['Test_Name'].str.contains('Uric Acid', case=False, na=False)].copy()
    # Convert object results to numeric, coercing strings like "Trace" or ">" to NaN
    uric['Test_Result'] = pd.to_numeric(uric['Test_Result'], errors='coerce')
    uric = uric.dropna(subset=['Test_Result', 'Laboratory_Delta'])
    
    # Find the measurement closest to Trial Day 0
    uric['abs_delta'] = uric['Laboratory_Delta'].abs()
    uric_base = uric.sort_values(['subject_id', 'abs_delta']).groupby('subject_id').first().reset_index()
    uric_base = uric_base.rename(columns={'Test_Result': 'Baseline_Uric_Acid'})[['subject_id', 'Baseline_Uric_Acid']]
    
    # Baseline FVC (% of Normal)
    df_fvc = dfs['fvc'].copy()
    df_fvc['pct_of_Normal_Trial_1'] = pd.to_numeric(df_fvc['pct_of_Normal_Trial_1'], errors='coerce')
    fvc = df_fvc.dropna(subset=['pct_of_Normal_Trial_1', 'Forced_Vital_Capacity_Delta']).copy()
    
    fvc['abs_delta'] = fvc['Forced_Vital_Capacity_Delta'].abs()
    fvc_base = fvc.sort_values(['subject_id', 'abs_delta']).groupby('subject_id').first().reset_index()
    fvc_base = fvc_base.rename(columns={'pct_of_Normal_Trial_1': 'Baseline_FVC_pct'})[['subject_id', 'Baseline_FVC_pct']]
    
    # -------------------------------------------------------------------------
    # 5. MERGE ALL MATRICES
    # -------------------------------------------------------------------------
    # Merge demographics
    df_model = df_model.merge(df_demo_clean, on='subject_id', how='left')
    
    # Merge history
    df_model = df_model.merge(df_hist_clean, on='subject_id', how='left')
    
    # Merge lab biomarkers
    df_model = df_model.merge(uric_base, on='subject_id', how='left')
    df_model = df_model.merge(fvc_base, on='subject_id', how='left')
    
    # Sort chronologically per patient for sequential processing in PyTorch
    df_model = df_model.sort_values(by=['subject_id', 'ALSFRS_Delta']).reset_index(drop=True)
    
    return df_model



In [5]:
# Execution Example:
processed_df = preprocess_proact_for_irt(datasets)
print(processed_df.info())
processed_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81041 entries, 0 to 81040
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   subject_id                     81041 non-null  int64  
 1   ALSFRS_Delta                   81041 non-null  float64
 2   Gastrostomy_Status             81041 non-null  int64  
 3   Q1_Speech                      79780 non-null  float64
 4   Q2_Salivation                  79778 non-null  float64
 5   Q3_Swallowing                  79777 non-null  float64
 6   Q4_Handwriting                 79774 non-null  float64
 7   Q5_Cutting                     79755 non-null  float64
 8   Q6_Dressing_and_Hygiene        79771 non-null  float64
 9   Q7_Turning_in_Bed              79771 non-null  float64
 10  Q8_Walking                     79774 non-null  float64
 11  Q9_Climbing_Stairs             79776 non-null  float64
 12  R_1_Dyspnea                    50657 non-null 

,subject_id,ALSFRS_Delta,Gastrostomy_Status,Q1_Speech,Q2_Salivation,Q3_Swallowing,Q4_Handwriting,Q5_Cutting,Q6_Dressing_and_Hygiene,Q7_Turning_in_Bed,...,Q9_Climbing_Stairs,R_1_Dyspnea,R_2_Orthopnea,R_3_Respiratory_Insufficiency,Age,Sex_Female,Bulbar_Onset,Diagnostic_Delay,Baseline_Uric_Acid,Baseline_FVC_pct
0,204,6.0,0,4.0,3.0,4.0,4.0,4.0,4.0,3.0,...,2.0,NaN,NaN,NaN,67.0,1.0,0.0,315.0,NaN,58.05
1,204,92.0,0,4.0,4.0,4.0,3.0,3.0,3.0,3.0,...,3.0,NaN,NaN,NaN,67.0,1.0,0.0,315.0,NaN,58.05
2,204,183.0,0,3.0,3.0,4.0,3.0,3.0,3.0,3.0,...,1.0,NaN,NaN,NaN,67.0,1.0,0.0,315.0,NaN,58.05
3,3301,5.0,0,3.0,4.0,3.0,3.0,3.0,2.0,3.0,...,1.0,3.0,4.0,4.0,NaN,0.0,0.0,NaN,279.556,NaN
4,3301,40.0,0,2.0,4.0,3.0,3.0,3.0,2.0,3.0,...,1.0,4.0,3.0,4.0,NaN,0.0,0.0,NaN,279.556,NaN
